# model

In [ ]:
# Refit all final-section models from raw data.

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import statsmodels.formula.api as smf
from patsy import bs
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

FINAL_SHP_PATH = r"../data/overall/admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp"
FINAL_REGION_PATH = r"../data/country/country_SSA_HE2.shp"
FINAL_OUT_DIR = r"./model_outputs"
os.makedirs(FINAL_OUT_DIR, exist_ok=True)

FINAL_HEAT_COL = "HE_perpop"
FINAL_FLOOD_COL = "sl_sevexp"
FINAL_EDI_COL = "EDI_qmean"
FINAL_FE_COL = "iso3"

final_gdf = gpd.read_file(FINAL_SHP_PATH)
final_gdf[FINAL_FE_COL] = final_gdf[FINAL_FE_COL].astype(str)

def final_zscore(x):
    x = pd.to_numeric(x, errors="coerce")
    std = x.std(ddof=0)
    if not np.isfinite(std) or std == 0:
        return x * np.nan
    return (x - x.mean()) / std

def final_fit_cluster(formula, data, weight_col):
    return smf.wls(formula, data=data, weights=data[weight_col]).fit(
        cov_type="cluster", cov_kwds={"groups": data[FINAL_FE_COL]}
    )

def final_fit_hc3(formula, data, weight_col):
    return smf.wls(formula, data=data, weights=data[weight_col]).fit(cov_type="HC3")

def final_collect(model, analysis, conversion, outcome, terms, nobs, note):
    rows = []
    for term in terms:
        if term not in model.params.index:
            continue
        coef = float(model.params[term])
        se = float(model.bse[term])
        pval = float(model.pvalues[term])
        rows.append({
            "analysis": analysis,
            "conversion": conversion,
            "outcome": outcome,
            "term": term,
            "n": int(nobs),
            "coef": coef,
            "se": se,
            "ci_low": coef - 1.96 * se,
            "ci_high": coef + 1.96 * se,
            "pval": pval,
            "positive": coef > 0,
            "p_lt_0_05": pval < 0.05,
            "note": note,
        })
    return rows

def final_prepare_ratio(numerator_col, denominator_col, conversion):
    d = final_gdf.copy()
    d["ratio_numer"] = pd.to_numeric(d[numerator_col], errors="coerce")
    d["ratio_denom"] = pd.to_numeric(d[denominator_col], errors="coerce")
    d["ratio_weight"] = d["ratio_denom"]
    d["conversion_ratio"] = d["ratio_numer"] / d["ratio_denom"]
    d.loc[(d["ratio_denom"] <= 0) | (d["ratio_numer"] < 0) | (d["conversion_ratio"] < 0), ["conversion_ratio", "ratio_weight"]] = np.nan
    d["log_ratio"] = np.log1p(d["conversion_ratio"])
    d["heat_z"] = final_zscore(d[FINAL_HEAT_COL])
    d["flood_z"] = final_zscore(d[FINAL_FLOOD_COL])
    d["edi_z"] = final_zscore(d[FINAL_EDI_COL])
    d["logpop_z"] = final_zscore(np.log1p(d["ratio_weight"]))
    d["high_heat75"] = (d[FINAL_HEAT_COL] >= d[FINAL_HEAT_COL].quantile(0.75)).astype(int)
    d["high_flood75"] = (d[FINAL_FLOOD_COL] >= d[FINAL_FLOOD_COL].quantile(0.75)).astype(int)
    d["joint_heat_flood75"] = ((d["high_heat75"] == 1) & (d["high_flood75"] == 1)).astype(int)
    d["conversion"] = conversion
    return d.dropna(subset=["log_ratio", "conversion_ratio", "ratio_weight", "heat_z", "flood_z", "edi_z", "logpop_z", FINAL_FE_COL]).copy()

final_iir = final_prepare_ratio("sl_pfinc", "sl_pfinf", "infection_to_incidence_ratio")
final_mir = final_prepare_ratio("sl_pfmort", "sl_pfinc", "incidence_to_mortality_ratio")

# Direct conversion-rate models.
final_main_rows = []
main_specs = [
    ("Continuous exposure main model", "log_ratio ~ heat_z + flood_z + heat_z:flood_z + edi_z + logpop_z", ["heat_z", "flood_z", "heat_z:flood_z", "edi_z", "logpop_z"]),
    ("Joint exposure main model", "log_ratio ~ high_heat75 + high_flood75 + joint_heat_flood75 + edi_z + logpop_z", ["high_heat75", "high_flood75", "joint_heat_flood75", "edi_z", "logpop_z"]),
]
for conversion, source_df in [("infection_to_incidence_ratio", final_iir), ("incidence_to_mortality_ratio", final_mir)]:
    for analysis, formula, terms in main_specs:
        model = final_fit_cluster(formula, source_df, "ratio_weight")
        final_main_rows += final_collect(model, analysis, conversion, "log1p(conversion_ratio)", terms, len(source_df), "WLS; denominator weights; country-clustered SE; no fixed effects")
final_main_summary = pd.DataFrame(final_main_rows)

# Grouped summaries for EDI and climate gradients.
def final_weighted_mean_ci(source_df, value_col, weight_col, group_col, order):
    rows = []
    for group in order:
        sub = source_df.loc[source_df[group_col].astype(str) == group].copy()
        x = pd.to_numeric(sub[value_col], errors="coerce")
        w = pd.to_numeric(sub[weight_col], errors="coerce")
        ok = x.notna() & w.notna() & (w > 0)
        x = x.loc[ok]
        w = w.loc[ok]
        if len(x) == 0:
            rows.append({"group": group, "n": 0, "weighted_mean": np.nan, "ci_low": np.nan, "ci_high": np.nan})
            continue
        mean = float(np.average(x, weights=w))
        se = float(np.sqrt(np.mean((x - mean) ** 2) / len(x)))
        rows.append({"group": group, "n": int(len(x)), "weighted_mean": mean, "ci_low": mean - 1.96 * se, "ci_high": mean + 1.96 * se})
    return pd.DataFrame(rows)

final_iir_group = final_iir.copy()
final_iir_group["edi_group"] = pd.qcut(final_iir_group[FINAL_EDI_COL], q=3, labels=["Low EDI", "Mid EDI", "High EDI"], duplicates="drop")
final_mir_group = final_mir.copy()
final_mir_group["climate_group"] = "Lower climate exposure"
final_mir_group.loc[(final_mir_group["high_heat75"] == 1) & (final_mir_group["high_flood75"] == 0), "climate_group"] = "High heat only"
final_mir_group.loc[(final_mir_group["high_heat75"] == 0) & (final_mir_group["high_flood75"] == 1), "climate_group"] = "High flood only"
final_mir_group.loc[final_mir_group["joint_heat_flood75"] == 1, "climate_group"] = "Joint high heat+flood"
# Build EDI and climate gradients for both conversion outcomes.
final_iir_group["climate_group"] = "Lower climate exposure"
final_iir_group.loc[(final_iir_group["high_heat75"] == 1) & (final_iir_group["high_flood75"] == 0), "climate_group"] = "High heat only"
final_iir_group.loc[(final_iir_group["high_heat75"] == 0) & (final_iir_group["high_flood75"] == 1), "climate_group"] = "High flood only"
final_iir_group.loc[final_iir_group["joint_heat_flood75"] == 1, "climate_group"] = "Joint high heat+flood"

final_mir_group["edi_group"] = pd.qcut(final_mir_group[FINAL_EDI_COL], q=3, labels=["Low EDI", "Mid EDI", "High EDI"], duplicates="drop")

final_group_summary = pd.concat([
    final_weighted_mean_ci(final_iir_group, "conversion_ratio", "ratio_weight", "edi_group", ["Low EDI", "Mid EDI", "High EDI"]).assign(panel="infection_to_incidence_by_EDI"),
    final_weighted_mean_ci(final_iir_group, "conversion_ratio", "ratio_weight", "climate_group", ["Lower climate exposure", "High heat only", "High flood only", "Joint high heat+flood"]).assign(panel="infection_to_incidence_by_climate"),
    final_weighted_mean_ci(final_mir_group, "conversion_ratio", "ratio_weight", "edi_group", ["Low EDI", "Mid EDI", "High EDI"]).assign(panel="incidence_to_mortality_by_EDI"),
    final_weighted_mean_ci(final_mir_group, "conversion_ratio", "ratio_weight", "climate_group", ["Lower climate exposure", "High heat only", "High flood only", "Joint high heat+flood"]).assign(panel="incidence_to_mortality_by_climate"),
], ignore_index=True)

# Climate-modified conversion slope models.
inc = final_gdf.copy()
inc["log_inc"] = np.log1p(pd.to_numeric(inc["sl_incrt"], errors="coerce"))
inc["pfpr_z"] = final_zscore(inc["sl_pfpr"])
inc["heat_z"] = final_zscore(inc[FINAL_HEAT_COL])
inc["flood_z"] = final_zscore(inc[FINAL_FLOOD_COL])
inc["info_z"] = final_zscore(inc[FINAL_EDI_COL])
inc["logpop_z"] = final_zscore(np.log1p(pd.to_numeric(inc["sl_incpop"], errors="coerce")))
inc = inc.dropna(subset=["log_inc", "pfpr_z", "heat_z", "flood_z", "info_z", "logpop_z", "sl_incpop"]).copy()
inc = inc[(inc["sl_incpop"] > 0) & (inc["sl_incrt"] >= 0) & (inc["sl_pfpr"] >= 0)].copy()

mort = final_gdf.copy()
mort["log_mort"] = np.log1p(pd.to_numeric(mort["sl_mortrt"], errors="coerce"))
mort["inc_z"] = final_zscore(mort["sl_incrt"])
mort["heat_z"] = final_zscore(mort[FINAL_HEAT_COL])
mort["flood_z"] = final_zscore(mort[FINAL_FLOOD_COL])
mort["info_z"] = final_zscore(mort[FINAL_EDI_COL])
mort["logpop_z"] = final_zscore(np.log1p(pd.to_numeric(mort["sl_morpop"], errors="coerce")))
mort = mort.dropna(subset=["log_mort", "inc_z", "heat_z", "flood_z", "info_z", "logpop_z", "sl_morpop"]).copy()
mort = mort[(mort["sl_morpop"] > 0) & (mort["sl_mortrt"] >= 0) & (mort["sl_incrt"] >= 0)].copy()

final_slope_rows = []
slope_specs = [
    ("PfPR x heat", "PfPR_to_incidence", inc, "sl_incpop", "log_inc", "log_inc ~ pfpr_z * heat_z + flood_z + info_z + logpop_z", "pfpr_z:heat_z"),
    ("PfPR x flood", "PfPR_to_incidence", inc, "sl_incpop", "log_inc", "log_inc ~ pfpr_z * flood_z + heat_z + info_z + logpop_z", "pfpr_z:flood_z"),
    ("Incidence x heat", "incidence_to_mortality", mort, "sl_morpop", "log_mort", "log_mort ~ inc_z * heat_z + flood_z + info_z + logpop_z", "inc_z:heat_z"),
    ("Incidence x flood", "incidence_to_mortality", mort, "sl_morpop", "log_mort", "log_mort ~ inc_z * flood_z + heat_z + info_z + logpop_z", "inc_z:flood_z"),
]
for analysis, conversion, source_df, weight_col, outcome, formula, term in slope_specs:
    model = final_fit_hc3(formula, source_df, weight_col)
    final_slope_rows += final_collect(model, analysis, conversion, outcome, [term], len(source_df), "Supplementary WLS HC3; no fixed effects")
final_slope_summary = pd.DataFrame(final_slope_rows)

# EDI spline models.
final_spline_rows = []
final_spline_predictions = []
for conversion, source_df in [("infection_to_incidence_ratio", final_iir), ("incidence_to_mortality_ratio", final_mir)]:
    linear_model = final_fit_cluster("log_ratio ~ edi_z + heat_z + flood_z + logpop_z", source_df, "ratio_weight")
    spline_model = final_fit_cluster("log_ratio ~ bs(edi_z, df=4, include_intercept=False) + heat_z + flood_z + logpop_z", source_df, "ratio_weight")
    wald = spline_model.wald_test_terms(skip_single=False).table.reset_index().rename(columns={"index": "term"})
    spline_term = wald[wald["term"].astype(str).str.contains("bs\\(edi_z", regex=True)]
    spline_p = float(spline_term["pvalue"].iloc[0]) if len(spline_term) and "pvalue" in spline_term.columns else np.nan
    final_spline_rows.append({"analysis": "EDI spline", "conversion": conversion, "n": int(len(source_df)), "linear_aic": float(linear_model.aic), "spline_aic": float(spline_model.aic), "linear_minus_spline_aic": float(linear_model.aic - spline_model.aic), "spline_term_pval": spline_p, "spline_better_by_aic": bool((linear_model.aic - spline_model.aic) > 2)})
    grid = np.linspace(source_df["edi_z"].quantile(0.02), source_df["edi_z"].quantile(0.98), 140)
    pred_df = pd.DataFrame({"edi_z": grid, "heat_z": 0.0, "flood_z": 0.0, "logpop_z": 0.0})
    pred = spline_model.get_prediction(pred_df).summary_frame(alpha=0.05)
    pred_df["fit"] = pred["mean"].values
    pred_df["ci_low"] = pred["mean_ci_lower"].values
    pred_df["ci_high"] = pred["mean_ci_upper"].values
    pred_df["conversion"] = conversion
    final_spline_predictions.append(pred_df)
final_spline_summary = pd.DataFrame(final_spline_rows)
final_spline_predictions = pd.concat(final_spline_predictions, ignore_index=True)


# Region-stratified EDI x climate moderation.
country_regions = gpd.read_file(FINAL_REGION_PATH)[["country_id", "country_na", "region"]].copy()
country_regions["country_id4"] = country_regions["country_id"].astype(str).str.extract(r"(\d{4})", expand=False)
region_map = country_regions.dropna(subset=["country_id4", "region"]).drop_duplicates("country_id4").set_index("country_id4")["region"]
region_lookup = pd.DataFrame(index=final_gdf.index)
region_lookup["country_id4"] = final_gdf["adm2ID"].astype(str).str.extract(r"(\d{4})", expand=False)
region_lookup["region"] = region_lookup["country_id4"].map(region_map)

def final_attach_region(source_df):
    out = source_df.copy()
    out["region"] = region_lookup.reindex(out.index)["region"].values
    return out.dropna(subset=["region"]).copy()

final_iir_region = final_attach_region(final_iir)
final_mir_region = final_attach_region(final_mir)

# Region-stratified EDI spline models.
REGION_ORDER = ["Western", "Central", "Eastern", "Northern", "Southern"]
final_region_spline_rows = []
final_region_spline_predictions = []
min_region_spline_n = 70

for conversion, source_df in [
    ("infection_to_incidence_ratio", final_iir_region),
    ("incidence_to_mortality_ratio", final_mir_region),
]:
    for region_name in REGION_ORDER:
        model_df = source_df[source_df["region"] == region_name].dropna(
            subset=["log_ratio", "ratio_weight", "edi_z", "heat_z", "flood_z", "logpop_z"]
        ).copy()
        row = {
            "analysis": "Region EDI spline",
            "conversion": conversion,
            "region": region_name,
            "n": int(len(model_df)),
            "linear_aic": np.nan,
            "spline_aic": np.nan,
            "linear_minus_spline_aic": np.nan,
            "spline_term_pval": np.nan,
            "spline_better_by_aic": False,
            "note": "Region-stratified WLS HC3; heat, flood, and log population held at 0 in predictions",
        }
        if len(model_df) < min_region_spline_n or model_df["edi_z"].nunique() < 6:
            row["note"] = f"Skipped: fewer than {min_region_spline_n} observations or insufficient EDI variation"
            final_region_spline_rows.append(row)
            continue
        try:
            linear_model = final_fit_hc3(
                "log_ratio ~ edi_z + heat_z + flood_z + logpop_z",
                model_df,
                "ratio_weight",
            )
            spline_model = final_fit_hc3(
                "log_ratio ~ bs(edi_z, df=4, include_intercept=False) + heat_z + flood_z + logpop_z",
                model_df,
                "ratio_weight",
            )
            wald = spline_model.wald_test_terms(skip_single=False).table.reset_index().rename(columns={"index": "term"})
            spline_term = wald[wald["term"].astype(str).str.contains("bs\\(edi_z", regex=True)]
            spline_p = float(spline_term["pvalue"].iloc[0]) if len(spline_term) and "pvalue" in spline_term.columns else np.nan
            row.update({
                "linear_aic": float(linear_model.aic),
                "spline_aic": float(spline_model.aic),
                "linear_minus_spline_aic": float(linear_model.aic - spline_model.aic),
                "spline_term_pval": spline_p,
                "spline_better_by_aic": bool((linear_model.aic - spline_model.aic) > 2),
            })

            grid_bounds = model_df["edi_z"].quantile([0.02, 0.98]).to_numpy(dtype=float)
            if not np.all(np.isfinite(grid_bounds)) or grid_bounds[0] >= grid_bounds[1]:
                row["note"] = "Skipped: invalid regional EDI prediction range"
                final_region_spline_rows.append(row)
                continue
            grid = np.linspace(grid_bounds[0], grid_bounds[1], 120)
            pred_df = pd.DataFrame({"edi_z": grid, "heat_z": 0.0, "flood_z": 0.0, "logpop_z": 0.0})
            pred = spline_model.get_prediction(pred_df).summary_frame(alpha=0.05)
            zero_idx = int(np.argmin(np.abs(pred_df["edi_z"].to_numpy())))
            base = float(pred["mean"].iloc[zero_idx])
            pred_out = pred_df[["edi_z"]].copy()
            pred_out["fit_rel"] = pred["mean"].to_numpy() - base
            pred_out["ci_low_rel"] = pred["mean_ci_lower"].to_numpy() - base
            pred_out["ci_high_rel"] = pred["mean_ci_upper"].to_numpy() - base
            pred_out["conversion"] = conversion
            pred_out["region"] = region_name
            pred_out["n"] = int(len(model_df))
            final_region_spline_predictions.append(pred_out)
        except Exception as exc:
            row["note"] = f"Skipped: {type(exc).__name__}: {exc}"
        final_region_spline_rows.append(row)

final_region_spline_summary = pd.DataFrame(final_region_spline_rows)
final_region_spline_predictions = (
    pd.concat(final_region_spline_predictions, ignore_index=True)
    if final_region_spline_predictions
    else pd.DataFrame(columns=["edi_z", "fit_rel", "ci_low_rel", "ci_high_rel", "conversion", "region", "n"])
)
region_specs = [
    ("Cont. EDI x heat", "log_ratio ~ edi_z * heat_z + flood_z + logpop_z", "edi_z:heat_z"),
    ("Cont. EDI x flood", "log_ratio ~ edi_z * flood_z + heat_z + logpop_z", "edi_z:flood_z"),
    ("Cont. EDI x heat:flood", "log_ratio ~ edi_z * heat_z * flood_z + logpop_z", "edi_z:heat_z:flood_z"),
    ("High EDI x heat", "log_ratio ~ edi_z * high_heat75 + edi_z * high_flood75 + edi_z * joint_heat_flood75 + logpop_z", "edi_z:high_heat75"),
    ("High EDI x flood", "log_ratio ~ edi_z * high_heat75 + edi_z * high_flood75 + edi_z * joint_heat_flood75 + logpop_z", "edi_z:high_flood75"),
    ("High EDI x joint heat+flood", "log_ratio ~ edi_z * high_heat75 + edi_z * high_flood75 + edi_z * joint_heat_flood75 + logpop_z", "edi_z:joint_heat_flood75"),
]
final_region_rows = []
min_region_n = 70
for conversion, source_df in [("infection_to_incidence_ratio", final_iir_region), ("incidence_to_mortality_ratio", final_mir_region)]:
    for region_name in sorted(source_df["region"].dropna().unique()):
        reg_df = source_df[source_df["region"] == region_name].copy()
        for analysis, formula, term in region_specs:
            needed = ["log_ratio", "ratio_weight", "edi_z", "heat_z", "flood_z", "logpop_z"]
            if analysis.startswith("High EDI"):
                needed += ["joint_heat_flood75", "high_heat75", "high_flood75"]
            model_df = reg_df.dropna(subset=needed).copy()
            if len(model_df) < min_region_n:
                final_region_rows.append({"region": region_name, "analysis": analysis, "conversion": conversion, "term": term, "n": int(len(model_df)), "coef": np.nan, "se": np.nan, "ci_low": np.nan, "ci_high": np.nan, "pval": np.nan, "positive": False, "p_lt_0_05": False})
                continue
            model = final_fit_hc3(formula, model_df, "ratio_weight")
            rows = final_collect(model, analysis, conversion, "log1p(conversion_ratio)", [term], len(model_df), "Region-stratified WLS HC3; no fixed effects")
            for row in rows:
                row["region"] = region_name
                final_region_rows.append(row)
final_region_summary = pd.DataFrame(final_region_rows)

# Save source data for the figure panels.
final_main_summary.to_csv(os.path.join(FINAL_OUT_DIR, "final_main_direct_models.csv"), index=False, encoding="utf-8-sig")
final_group_summary.to_csv(os.path.join(FINAL_OUT_DIR, "final_grouped_summaries.csv"), index=False, encoding="utf-8-sig")
final_slope_summary.to_csv(os.path.join(FINAL_OUT_DIR, "final_slope_models.csv"), index=False, encoding="utf-8-sig")
final_spline_summary.to_csv(os.path.join(FINAL_OUT_DIR, "final_spline_summary.csv"), index=False, encoding="utf-8-sig")
final_spline_predictions.to_csv(os.path.join(FINAL_OUT_DIR, "final_spline_predictions.csv"), index=False, encoding="utf-8-sig")
final_region_spline_summary.to_csv(os.path.join(FINAL_OUT_DIR, "final_region_spline_summary.csv"), index=False, encoding="utf-8-sig")
final_region_spline_predictions.to_csv(os.path.join(FINAL_OUT_DIR, "final_region_spline_predictions.csv"), index=False, encoding="utf-8-sig")
final_region_summary.to_csv(os.path.join(FINAL_OUT_DIR, "final_region_moderation.csv"), index=False, encoding="utf-8-sig")

print("Final section models refit from raw data.")
print("Main model terms:")
print(final_main_summary[["analysis", "conversion", "term", "n", "coef", "ci_low", "ci_high", "pval", "positive", "p_lt_0_05"]].to_string(index=False))
print("\nSlope models:")
print(final_slope_summary[["analysis", "conversion", "term", "n", "coef", "ci_low", "ci_high", "pval", "positive", "p_lt_0_05"]].to_string(index=False))
print("\nSpline summary:")
print(final_spline_summary.to_string(index=False))
print("\nRegion spline summary:")
print(final_region_spline_summary[["conversion", "region", "n", "linear_minus_spline_aic", "spline_term_pval", "spline_better_by_aic", "note"]].to_string(index=False))
print("\nRegion moderation rows:", len(final_region_summary))
print("Saved source data to:", FINAL_OUT_DIR)


# figure2a&3a

In [ ]:
# Shared visualization setup.

FINAL_OUT_DIR = r"."
os.makedirs(FINAL_OUT_DIR, exist_ok=True)

required_objects = [
    "FINAL_OUT_DIR", "FINAL_EDI_COL", "final_main_summary", "final_group_summary",
    "final_spline_predictions", "final_region_spline_predictions", "final_region_spline_summary",
    "final_iir_region", "final_mir_region", "final_region_summary",
]
missing = [name for name in required_objects if name not in globals()]
if missing:
    raise RuntimeError(f"Run the previous final section model cell first; missing: {missing}")

mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]

mpl.rcParams.update({
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.size": 7,
    "axes.titlesize": 7,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 6,
    "axes.linewidth": 0.55,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.major.width": 0.55,
    "ytick.major.width": 0.55,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
})

COL_EDI = "#59B2E1"
COL_CLIM = "#DF80BF"
COL_GREY = "#6F6F6F"
COL_DARK = "#2B2B2B"
COL_LIGHT = "#ECECEC"

def final_pct(coef):
    return 100 * (np.exp(coef) - 1)

def final_save(fig, base_name):
    base = os.path.join(FINAL_OUT_DIR, base_name)
    fig.savefig(base + ".pdf", bbox_inches="tight")
    fig.savefig(base + ".svg", bbox_inches="tight")
    fig.savefig(base + ".png", dpi=600, bbox_inches="tight")

# Direct conversion-rate model panels.
def plot_direct_model(ax, df, title):
    model_specs = [
        (
            "Continuous exposure",
            "Continuous exposure main model",
            [
                ("edi_z", "EDI (+1 s.d.)", COL_EDI),
                ("heat_z", "Heat (+1 s.d.)", COL_CLIM),
                ("flood_z", "Flood (+1 s.d.)", COL_CLIM),
                ("heat_z:flood_z", "Heat x flood", COL_CLIM),
            ],
        ),
        (
            "High exposure",
            "Joint exposure main model",
            [
                ("edi_z", "EDI (+1 s.d.)", COL_EDI),
                ("high_heat75", "High heat", COL_CLIM),
                ("high_flood75", "High flood", COL_CLIM),
                ("joint_heat_flood75", "Joint high\n heat + flood", COL_CLIM),
            ],
        ),
    ]

    rows = []
    y = 0.0
    for group_label, analysis_label, terms in model_specs:
        for term, label, color in terms:
            sub = df[(df["analysis"] == analysis_label) & (df["term"] == term)]
            if sub.empty:
                continue
            row = sub.iloc[0].copy()
            row["plot_y"] = y
            row["plot_label"] = label
            row["plot_color"] = color
            rows.append(row)
            y += 1.0
        y += 0.75

    d = pd.DataFrame(rows)
    d["pct"] = d["coef"].map(final_pct)
    d["ci_low_pct"] = d["ci_low"].map(final_pct)
    d["ci_high_pct"] = d["ci_high"].map(final_pct)

    x_min = min(float(d["ci_low_pct"].min()), float(d["pct"].min()), 0.0)
    x_max = max(float(d["ci_high_pct"].max()), float(d["pct"].max()), 0.0)
    x_span = max(x_max - x_min, 1.0)
    x_pad = max(0.12 * x_span, 0.02)
    label_pad = max(0.020 * x_span, 0.012)
    ax.set_xlim(x_min - 1.65 * x_pad, x_max + 1.85 * x_pad)
    ax.axvline(0, color="0.35", lw=0.7, zorder=1)

    for _, row in d.iterrows():
        x = float(row["pct"])
        lo = float(row["ci_low_pct"])
        hi = float(row["ci_high_pct"])
        y0 = float(row["plot_y"])
        left = min(0.0, x)
        width = abs(x)

        ax.barh(
            y0,
            width,
            left=left,
            height=0.56,
            color=row["plot_color"],
            alpha=0.72,
            edgecolor=row["plot_color"],
            linewidth=0.8,
            zorder=2,
        )
        ax.errorbar(
            x,
            y0,
            xerr=[[x - lo], [hi - x]],
            fmt="none",
            ecolor=COL_DARK,
            elinewidth=0.9,
            capsize=2.4,
            capthick=0.9,
            zorder=3,
        )
        if x >= 0:
            text_x = hi + label_pad
            ha = "left"
        else:
            text_x = lo - label_pad
            ha = "right"
        sig = "* " if bool(row["p_lt_0_05"]) else ""
        ax.text(
            text_x,
            y0,
            f"{sig}{x:+.2f}%",
            fontsize=7,
            ha=ha,
            va="center",
            color=COL_DARK,
        )

    group_gap_y = 3.875
    ax.axhline(group_gap_y, color="0.78", lw=0.7, zorder=0, ls="--")
    x_left, x_right = ax.get_xlim()
    x_text = x_left + 0.01 * (x_right - x_left)
    ax.text(x_text, -0.68, "Continuous exposure", fontsize=7, ha="left", va="center", color=COL_DARK)
    ax.text(x_text, group_gap_y + 0.43, "High exposure", fontsize=7, ha="left", va="center", color=COL_DARK)

    ax.set_yticks(d["plot_y"].to_numpy())
    ax.set_yticklabels(d["plot_label"].tolist())
    ax.set_ylim(float(d["plot_y"].max()) + 0.65, -0.95)
    ax.set_xlabel("Change in 1+burden ratio (%)")
    ax.set_title(title, fontsize=8)
    ax.grid(axis="x", color=COL_LIGHT, lw=0.5, zorder=0)
    ax.text(0.985, 0.965, "* p < 0.05", transform=ax.transAxes, ha="right", va="top", fontsize=7, color=COL_DARK)

for conversion, title, base_name in [
    ("infection_to_incidence_ratio", "Infection-to-incidence conversion", "figure2a"),
    ("incidence_to_mortality_ratio", "Incidence-to-mortality conversion", "figure3a"),
]:
    fig, ax = plt.subplots(figsize=(6, 4))
    main = final_main_summary[final_main_summary["conversion"] == conversion].copy()
    plot_direct_model(ax, main, title)
    fig.subplots_adjust(left=0.29, right=0.98, top=0.90, bottom=0.16)
    final_save(fig, base_name)
    plt.show()


# figure2d&3d

In [ ]:
# Regional EDI moderation maps with bar glyphs.
import geopandas as gpd
from matplotlib.patches import Patch
from shapely.geometry import GeometryCollection, MultiPolygon, Polygon

def _polygon_exteriors_only(geom):
    """Remove holes/interior rings so the basemap only shows region outlines."""
    if geom is None or geom.is_empty:
        return geom
    geom = geom.buffer(0)
    if isinstance(geom, Polygon):
        return Polygon(geom.exterior)
    if isinstance(geom, MultiPolygon):
        return MultiPolygon([Polygon(poly.exterior) for poly in geom.geoms if not poly.is_empty])
    if isinstance(geom, GeometryCollection):
        polygons = []
        for part in geom.geoms:
            cleaned = _polygon_exteriors_only(part)
            if isinstance(cleaned, Polygon):
                polygons.append(cleaned)
            elif isinstance(cleaned, MultiPolygon):
                polygons.extend(list(cleaned.geoms))
        return MultiPolygon(polygons) if polygons else geom
    return geom

def build_region_base_map():
    region_gdf = gpd.read_file(FINAL_REGION_PATH)[["region", "geometry"]].copy()
    region_gdf = region_gdf.dropna(subset=["region"]).to_crs("EPSG:4326")
    region_gdf = region_gdf.dissolve(by="region", as_index=False)
    region_gdf["geometry"] = region_gdf.geometry.map(_polygon_exteriors_only)
    return region_gdf

def build_country_boundary_map():
    country_gdf = gpd.read_file(FINAL_REGION_PATH)[["geometry"]].copy()
    country_gdf = country_gdf.dropna(subset=["geometry"]).to_crs("EPSG:4326")
    country_gdf["geometry"] = country_gdf.geometry.map(_polygon_exteriors_only)
    return country_gdf.boundary

def plot_bar_glyph(ax, x0, y0, effects, pvals, vmax, bar_height=7.6, bar_width=0.68):
    term_specs = [
        ("Cont. EDI x heat", -2.25, "#F5994F", None),
        ("Cont. EDI x flood", -1.35, "#6DA7D1", None),
        ("Cont. EDI x heat:flood", -0.45, COL_CLIM, None),
        ("High EDI x heat", 0.75, "#F5994F", "/////"),
        ("High EDI x flood", 1.65, "#6DA7D1", "/////"),
        ("High EDI x joint heat+flood", 2.55, COL_CLIM, "/////"),
    ]
    ax.plot([x0 - 2.60, x0 + 2.90], [y0, y0], color="0.25", lw=0.55, zorder=5)
    ax.plot([x0 + 0.15, x0 + 0.15], [y0 - 0.38, y0 + 0.38], color="0.45", lw=0.45, zorder=5)
    for term, offset, color, hatch in term_specs:
        val = effects.get(term, np.nan)
        if not np.isfinite(val):
            continue
        scaled = np.sign(val) * np.sqrt(min(abs(val) / vmax, 1.0))
        height = scaled * bar_height
        x = x0 + offset
        bottom = y0 if height >= 0 else y0 + height
        ax.bar(
            x,
            abs(height),
            width=bar_width,
            bottom=bottom,
            color=color,
            edgecolor="0.34" if hatch else "white",
            linewidth=0.32 if hatch else 0.22,
            hatch=hatch,
            zorder=6,
        )
        if pvals.get(term, False):
            star_y = y0 + height + (0.72 if height >= 0 else -0.72)
            ax.text(x, star_y, "*", ha="center", va="center", fontsize=6.4, fontweight="bold", color=COL_DARK, zorder=8)

def plot_region_moderation_single(ax, conversion, title):
    region_map = build_region_base_map()
    country_boundary = build_country_boundary_map()
    d = final_region_summary[final_region_summary["conversion"] == conversion].copy()
    analysis_order = ["Cont. EDI x heat", "Cont. EDI x flood", "Cont. EDI x heat:flood", "High EDI x heat", "High EDI x flood", "High EDI x joint heat+flood"]
    region_order = ["Western", "Central", "Eastern", "Northern", "Southern"]
    d = d[d["analysis"].isin(analysis_order) & d["region"].isin(region_order)].copy()
    d["effect_pct"] = 100 * (np.exp(d["coef"]) - 1)

    vals = d["effect_pct"].replace([np.inf, -np.inf], np.nan).dropna()
    vmax = max(float(vals.abs().quantile(0.90)), 0.05) if len(vals) else 1.0
    vmax = min(vmax, 8.0)

    region_map.plot(ax=ax, color="#F2F2F2", edgecolor="none", linewidth=0, zorder=1)
    country_boundary.plot(ax=ax, color="0.80", linewidth=0.24, zorder=2)
    region_map.boundary.plot(ax=ax, color="0.52", linewidth=0.62, zorder=3)

    anchors = {
        "Western": (-1, 12.0),
        "Central": (19.0, 1.5),
        "Eastern": (36.0, -3),
        "Northern": (30.0, 16.8),
        "Southern": (23.0, -25.5),
    }

    for region in region_order:
        sub = d[d["region"] == region].copy()
        if sub.empty:
            continue
        effects = dict(zip(sub["analysis"], sub["effect_pct"]))
        pvals = dict(zip(sub["analysis"], sub["p_lt_0_05"]))
        x0, y0 = anchors[region]
        plot_bar_glyph(ax, x0, y0, effects, pvals, vmax=vmax)

    ax.set_title(title, fontsize=9, pad=1)
    ax.set_axis_off()
    ax.set_xlim(-20, 55)
    ax.set_ylim(-37, 29)
    ax.set_aspect("equal")

    plt.rcParams["hatch.linewidth"] = 0.35

    handles = [
        Patch(facecolor="#F5994F", edgecolor="white", label="Cont: EDI x heat"),
        Patch(facecolor="#6DA7D1", edgecolor="white", label="Cont: EDI x flood"),
        Patch(facecolor=COL_CLIM, edgecolor="white", label="Cont: EDI x heat:flood"),
        Patch(facecolor="#F5994F", edgecolor="0.5", linewidth=0.35, hatch="/////", label="High: EDI x heat"),
        Patch(facecolor="#6DA7D1", edgecolor="0.5", linewidth=0.35, hatch="/////", label="High: EDI x flood"),
        Patch(facecolor=COL_CLIM, edgecolor="0.5", linewidth=0.35, hatch="/////", label="High: EDI x joint"),
    ]
    ax.legend(handles=handles, loc="lower left", bbox_to_anchor=(0.01, 0.015), frameon=False, fontsize=6, ncol=2, handlelength=0.8, handletextpad=0.32, columnspacing=0.75)
    ax.text(
        0.01,
        0.145,
        f"Bar height: signed sqrt-scaled coefficient (%)\nReference scale: +/-{vmax:.2f}%\n* p < 0.05",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=6,
        color=COL_DARK,
    )

for conversion, title, base_name in [
    ("infection_to_incidence_ratio", "Regional EDI moderation: infection-to-incidence", "figure2d"),
    ("incidence_to_mortality_ratio", "Regional EDI moderation: incidence-to-mortality", "figure3d"),
]:
    fig, ax = plt.subplots(figsize=(5.2, 7.4))
    plot_region_moderation_single(ax, conversion, title)
    fig.subplots_adjust(left=0.02, right=0.98, top=0.98, bottom=0.04)
    final_save(fig, base_name)
    plt.show()


# figure2b&3b

In [ ]:
# Region-stratified nonlinear EDI association panels.
REGION_ORDER = ["Western", "Central", "Eastern", "Northern", "Southern"]
REGION_COLORS = {
    "Western": "#F1D8C7",
    "Central": "#F2ACB8",
    "Eastern": "#F38D76",
    "Northern": "#F0A35B",
    "Southern": "#F4C76A",
}


def _region_source_for_conversion(conversion):
    if conversion == "infection_to_incidence_ratio":
        return final_iir_region
    if conversion == "incidence_to_mortality_ratio":
        return final_mir_region
    raise ValueError(f"Unknown conversion: {conversion}")


def _global_spline_relative(conversion):
    sub = final_spline_predictions[
        final_spline_predictions["conversion"] == conversion
    ].sort_values("edi_z").copy()
    zero_idx = int(np.argmin(np.abs(sub["edi_z"].to_numpy())))
    base = float(sub["fit"].iloc[zero_idx])
    sub["fit_rel"] = sub["fit"] - base
    sub["ci_low_rel"] = sub["ci_low"] - base
    sub["ci_high_rel"] = sub["ci_high"] - base
    return sub



def final_save_spline_panel(fig, base_name):
    base = os.path.join(FINAL_OUT_DIR, base_name)
    fig.savefig(base + ".pdf", bbox_inches="tight")
    fig.savefig(base + ".svg", bbox_inches="tight")
    fig.savefig(base + ".png", dpi=600, bbox_inches="tight")


def _region_labels(conversion):
    d = final_region_spline_summary[
        final_region_spline_summary["conversion"] == conversion
    ].copy()
    return {
        row["region"]: row["region"]
        for _, row in d.iterrows()
        if row["region"] in REGION_ORDER and int(row["n"]) > 0
    }


def plot_spline_single(ax, conversion, title, title_fontsize=10.5, title_pad=5):
    overall = _global_spline_relative(conversion)
    labels = _region_labels(conversion)
    region_pred = final_region_spline_predictions[
        final_region_spline_predictions["conversion"] == conversion
    ].copy()

    y_parts = [overall[["ci_low_rel", "ci_high_rel"]].to_numpy().ravel()]
    if not region_pred.empty:
        y_parts.append(region_pred["fit_rel"].to_numpy())
    y_vals = pd.Series(np.concatenate(y_parts)).replace([np.inf, -np.inf], np.nan).dropna()
    y_min, y_max = float(y_vals.quantile(0.02)), float(y_vals.quantile(0.98))
    if not np.isfinite(y_min) or not np.isfinite(y_max) or y_min >= y_max:
        y_min, y_max = -0.1, 0.1
    y_span = y_max - y_min
    ax.set_ylim(y_min - 0.08 * y_span, y_max + 0.10 * y_span)

    ax.fill_between(
        overall["edi_z"].to_numpy(),
        overall["ci_low_rel"].to_numpy(),
        overall["ci_high_rel"].to_numpy(),
        color="0.70",
        alpha=0.18,
        lw=0,
        zorder=1,
    )
    ax.plot(
        overall["edi_z"],
        overall["fit_rel"],
        color="0.30",
        lw=1.35,
        ls="--",
        label="Overall",
        zorder=9,
    )

    for region in REGION_ORDER:
        sub = region_pred[region_pred["region"] == region].sort_values("edi_z")
        if sub.empty:
            continue
        color = REGION_COLORS[region]
        ax.plot(
            sub["edi_z"],
            sub["fit_rel"],
            color=color,
            lw=1.55,
            label=labels.get(region, region),
            zorder=4,
        )

    ax.axhline(0, color="0.58", lw=0.55, zorder=0)
    ax.set_xlabel("EDI z-score", fontsize=9.5)
    ax.set_ylabel("Predicted log ratio\n(relative to EDI = 0)", fontsize=9.5)
    ax.set_title(title, fontsize=title_fontsize, pad=title_pad)
    ax.tick_params(axis="both", labelsize=8.5)
    ax.grid(axis="y", color=COL_LIGHT, lw=0.5)
    # ax.text(
    #     0.99,
    #     0.025,
    #     "Grey band: overall 95% CI",
    #     transform=ax.transAxes,
    #     fontsize=7.2,
    #     color="0.35",
    #     ha="right",
    #     va="bottom",
    # )
    ax.legend(
        loc="upper left",
        bbox_to_anchor=(0.012, 0.988),
        frameon=False,
        fontsize=8,
        ncol=2,
        handlelength=1.15,
        handletextpad=0.28,
        columnspacing=0.55,
        labelspacing=0.16,
        borderaxespad=0,
    )


for conversion, title, base_name, title_fontsize, title_pad in [
    ("infection_to_incidence_ratio", "Region-stratified nonlinear EDI association:\ninfection-to-incidence", "figure2b", 12.0, 9),
    ("incidence_to_mortality_ratio", "Region-stratified nonlinear EDI association:\nincidence-to-mortality", "figure3b", 12.0, 9),
]:
    fig, ax = plt.subplots(figsize=(4, 4))
    plot_spline_single(ax, conversion, title, title_fontsize=title_fontsize, title_pad=title_pad)
    fig.subplots_adjust(left=0.17, right=0.97, top=0.88, bottom=0.16)
    final_save_spline_panel(fig, base_name)
    plt.show()


# figure2c&3c

In [ ]:
# Region-stacked climate exposure gradient panels.
REGION_ORDER = ["Western", "Central", "Eastern", "Northern", "Southern"]
REGION_COLORS_STACKED = {
    "Western": "#F1D8C7",
    "Central": "#F2ACB8",
    "Eastern": "#F38D76",
    "Northern": "#F0A35B",
    "Southern": "#F4C76A",
}


def _add_climate_group(source_df):
    out = source_df.copy()
    out["climate_group"] = "Lower climate exposure"
    out.loc[
        (out["high_heat75"] == 1) & (out["high_flood75"] == 0),
        "climate_group",
    ] = "High heat only"
    out.loc[
        (out["high_heat75"] == 0) & (out["high_flood75"] == 1),
        "climate_group",
    ] = "High flood only"
    out.loc[out["joint_heat_flood75"] == 1, "climate_group"] = "Joint high heat+flood"
    return out


def _region_stacked_components(source_df, order, scale):
    d = _add_climate_group(source_df)
    d = d.dropna(subset=["climate_group", "region", "conversion_ratio", "ratio_weight"]).copy()
    d = d[d["climate_group"].isin(order) & d["region"].isin(REGION_ORDER)].copy()
    d["weighted_value"] = d["conversion_ratio"] * d["ratio_weight"]

    rows = []
    for group in order:
        group_df = d[d["climate_group"] == group].copy()
        total_weight = float(group_df["ratio_weight"].sum())
        if total_weight <= 0:
            continue
        for region in REGION_ORDER:
            reg_df = group_df[group_df["region"] == region]
            contribution = float(reg_df["weighted_value"].sum()) / total_weight * scale
            rows.append({
                "group": group,
                "region": region,
                "contribution": contribution,
            })
    return pd.DataFrame(rows)


def _source_for_climate_panel(panel_name):
    if panel_name == "infection_to_incidence_by_climate":
        return final_iir_region
    if panel_name == "incidence_to_mortality_by_climate":
        return final_mir_region
    raise ValueError(f"Unknown climate panel: {panel_name}")


def _format_region_component_label(value, scale):
    if scale >= 100:
        return f"{value:.1f}"
    return f"{value:.2f}"


def plot_climate_gradient_single(ax, panel_name, ylabel, title, scale=1.0, y_floor=None):
    order = ["Lower climate exposure", "High heat only", "High flood only", "Joint high heat+flood"]
    labels = ["Lower\nexposure", "High\nheat", "High\nflood", "Joint high\nexposure"]

    d = final_group_summary[final_group_summary["panel"] == panel_name].copy()
    d["x"] = d["group"].map({g: i for i, g in enumerate(order)})
    d = d.dropna(subset=["x"]).sort_values("x").copy()
    d["mean_plot"] = d["weighted_mean"] * scale
    d["ci_low_plot"] = d["ci_low"] * scale
    d["ci_high_plot"] = d["ci_high"] * scale

    components = _region_stacked_components(_source_for_climate_panel(panel_name), order, scale)
    component_pivot = (
        components.pivot(index="group", columns="region", values="contribution")
        .reindex(order)
        .reindex(columns=REGION_ORDER)
        .fillna(0.0)
    )

    baseline = float(d.loc[d["group"] == "Lower climate exposure", "mean_plot"].iloc[0])
    y = d["mean_plot"].to_numpy()
    yerr_low = (d["mean_plot"] - d["ci_low_plot"]).to_numpy()
    yerr_high = (d["ci_high_plot"] - d["mean_plot"]).to_numpy()
    span = max(float(d["ci_high_plot"].max() - min(0, d["ci_low_plot"].min())), 1e-9)
    segment_label_min = 0.070 * span

    ax.axhline(baseline, color="0.45", lw=0.7, ls=(0, (2.0, 2.0)), zorder=0)
    x_positions = np.arange(len(order), dtype=float)
    bottoms = np.zeros(len(order), dtype=float)
    for region in REGION_ORDER:
        heights = component_pivot[region].to_numpy(dtype=float)
        ax.bar(
            x_positions,
            heights,
            width=0.62,
            bottom=bottoms,
            color=REGION_COLORS_STACKED[region],
            edgecolor="white",
            linewidth=0.45,
            alpha=0.92,
            label=region,
            zorder=2,
        )
        for x_pos, bottom, height in zip(x_positions, bottoms, heights):
            if height <= 0 or height < segment_label_min:
                continue
            ax.text(
                x_pos,
                bottom + 0.5 * height,
                _format_region_component_label(height, scale),
                ha="center",
                va="center",
                fontsize=8,
                color=COL_DARK,
                zorder=5,
                clip_on=True,
            )
        bottoms += heights

    ax.errorbar(
        d["x"],
        y,
        yerr=[yerr_low, yerr_high],
        fmt="o",
        ms=4.3,
        capsize=2.6,
        lw=0.85,
        color=COL_DARK,
        mfc="white",
        mec=COL_DARK,
        mew=0.8,
        zorder=4,
    )
    ax.plot(d["x"], y, color="0.50", lw=0.65, zorder=3)

    y_top = float(d["ci_high_plot"].max() + 0.28 * span)
    auto_bottom = float(d["ci_low_plot"].min() - 0.10 * span)
    y_bottom = y_floor if y_floor is not None else auto_bottom
    ax.set_ylim(y_bottom, y_top)

    for _, row in d.iterrows():
        x = float(row["x"])
        mean = float(row["mean_plot"])
        ci_hi = float(row["ci_high_plot"])
        if row["group"] == "Lower climate exposure":
            label = "ref."
            txt_color = "0.35"
        else:
            diff_pct = 100 * (mean / baseline - 1) if baseline != 0 else np.nan
            label = f"{diff_pct:+.1f}%"
            txt_color = COL_DARK
        ax.text(
            x,
            ci_hi + 0.055 * span,
            label,
            ha="center",
            va="bottom",
            fontsize=9,
            color=txt_color,
        )

    ax.set_xticks(np.arange(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_xlabel("Climate exposure group", fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=11, pad=6)
    ax.grid(axis="y", color=COL_LIGHT, lw=0.5, zorder=0)
    ax.tick_params(axis="both", labelsize=9)
    ax.legend(
        loc="upper left",
        bbox_to_anchor=(0.0, 1.0),
        frameon=False,
        fontsize=8,
        ncol=2,
        handlelength=1.2,
        handletextpad=0.35,
        columnspacing=0.55,
        labelspacing=0.20,
        borderaxespad=0.0,
    )


def final_save_climate_panel(fig, base_name):
    base = os.path.join(FINAL_OUT_DIR, base_name)
    fig.savefig(base + ".pdf", bbox_inches="tight")
    fig.savefig(base + ".svg", bbox_inches="tight")
    fig.savefig(base + ".png", dpi=600, bbox_inches="tight")


for panel_name, ylabel, scale, y_floor, title, base_name in [
    ("infection_to_incidence_by_climate", "Incident cases per infection", 1.0, 0.0, "Climate exposure gradient:\ninfection-to-incidence", "figure2c"),
    ("incidence_to_mortality_by_climate", "Deaths per 1,000 incident cases", 1000.0, 0.0, "Climate exposure gradient:\nincidence-to-mortality", "figure3c"),
]:
    fig, ax = plt.subplots(figsize=(4, 4))
    plot_climate_gradient_single(ax, panel_name, ylabel, title, scale=scale, y_floor=y_floor)
    fig.subplots_adjust(left=0.16, right=0.97, top=0.87, bottom=0.20)
    final_save_climate_panel(fig, base_name)
    plt.show()


# figure2e&3e

In [ ]:
# Selected standardized contribution maps at ADM2 scale.
SELECTED_MAP_OUT_DIR = os.path.join(FINAL_OUT_DIR)
os.makedirs(SELECTED_MAP_OUT_DIR, exist_ok=True)

CONT_MAP_FORMULA = "log_ratio_z ~ heat_z + flood_z + heat_flood_z + edi_z + logpop_z"
HIGH_MAP_FORMULA = "log_ratio_z ~ high_heat75 + high_flood75 + joint_heat_flood75 + edi_z + logpop_z"

def selected_model_component_sd(model, df, terms):
    comp = np.zeros(len(df), dtype=float)
    for term in terms:
        if term in model.params.index and term in df.columns:
            comp += (
                float(model.params[term])
                * pd.to_numeric(df[term], errors="coerce").fillna(0).to_numpy()
            )
    return comp

def build_selected_map_fields(source_df, conversion_label):
    out = source_df.copy()
    out["log_ratio_z"] = final_zscore(out["log_ratio"])
    out["heat_flood_z"] = (
        pd.to_numeric(out["heat_z"], errors="coerce")
        * pd.to_numeric(out["flood_z"], errors="coerce")
    )

    cont_model = final_fit_cluster(CONT_MAP_FORMULA, out, "ratio_weight")
    high_model = final_fit_cluster(HIGH_MAP_FORMULA, out, "ratio_weight")

    out["cont_edi_contribution_sd"] = selected_model_component_sd(cont_model, out, ["edi_z"])
    out["cont_flood_contribution_sd"] = selected_model_component_sd(cont_model, out, ["flood_z"])
    out["high_edi_contribution_sd"] = selected_model_component_sd(high_model, out, ["edi_z"])
    out["high_heat_contribution_sd"] = selected_model_component_sd(high_model, out, ["high_heat75"])
    out["high_flood_contribution_sd"] = selected_model_component_sd(high_model, out, ["high_flood75"])
    out["high_joint_contribution_sd"] = selected_model_component_sd(high_model, out, ["joint_heat_flood75"])
    out["map_conversion"] = conversion_label
    return out, cont_model, high_model

def selected_common_symmetric_limits(map_frames, cols, q=0.98):
    vals = []
    for gdf in map_frames:
        for col in cols:
            vals.append(pd.to_numeric(gdf[col], errors="coerce"))
    vals = pd.concat(vals).replace([np.inf, -np.inf], np.nan).dropna()
    if vals.empty:
        return -1.0, 1.0
    vmax = max(float(vals.abs().quantile(q)), 1e-6)
    return -vmax, vmax

def build_selected_map_boundaries():
    country_gdf = gpd.read_file(FINAL_REGION_PATH)[["geometry"]].copy()
    country_gdf = country_gdf.dropna(subset=["geometry"]).to_crs("EPSG:4326")
    country_boundary = country_gdf.boundary
    ssa_boundary = country_gdf.dissolve().boundary
    return country_boundary, ssa_boundary

selected_country_boundary, selected_ssa_boundary = build_selected_map_boundaries()

def plot_selected_contribution_row(map_gdf, panels, title, base_name, clim):
    plot_gdf = map_gdf.copy()
    if plot_gdf.crs is not None:
        plot_gdf = plot_gdf.to_crs("EPSG:4326")

    fig, axes = plt.subplots(1, 3, figsize=(10.2, 3.35))
    vmin, vmax = clim

    for ax, (col, panel_title) in zip(axes, panels):
        plot_gdf.plot(
            column=col,
            ax=ax,
            cmap="RdBu_r",
            vmin=vmin,
            vmax=vmax,
            linewidth=0.03,
            edgecolor="0.88",
            missing_kwds={"color": "0.92", "edgecolor": "0.88", "linewidth": 0.03},
            zorder=1,
        )
        selected_country_boundary.plot(ax=ax, color="0.38", linewidth=0.18, zorder=3)
        selected_ssa_boundary.plot(ax=ax, color="0.18", linewidth=0.58, zorder=4)
        ax.set_title(panel_title, fontsize=8)
        ax.set_axis_off()

    norm = mpl.colors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=mpl.colormaps.get_cmap("RdBu_r"))
    sm.set_array([])
    cax = fig.add_axes([0.948, 0.25, 0.012, 0.48])
    cbar = fig.colorbar(sm, cax=cax, orientation="vertical")
    cbar.set_label("Model-attributed contribution (SD)", fontsize=7, labelpad=5)
    cbar.ax.tick_params(labelsize=5.6, length=2)
    cbar.outline.set_linewidth(0.35)

    fig.suptitle(title, x=0.02, y=0.99, ha="left", fontsize=9.0, fontweight="bold")
    fig.subplots_adjust(left=0.015, right=0.925, top=0.88, bottom=0.045, wspace=0.005)

    out_base = os.path.join(SELECTED_MAP_OUT_DIR, base_name)
    fig.savefig(out_base + ".pdf", bbox_inches="tight")
    plt.show()

selected_iir_map, selected_iir_cont_model, selected_iir_high_model = build_selected_map_fields(
    final_iir,
    "infection_to_incidence_ratio",
)
selected_mir_map, selected_mir_cont_model, selected_mir_high_model = build_selected_map_fields(
    final_mir,
    "incidence_to_mortality_ratio",
)

selected_iir_panels = [
    ("cont_edi_contribution_sd", "Continuous exposure model: EDI"),
    ("high_edi_contribution_sd", "High-exposure model: EDI"),
    ("high_heat_contribution_sd", "High-exposure model: high heat"),
]
selected_mir_panels = [
    ("cont_flood_contribution_sd", "Continuous exposure model: flood"),
    ("high_flood_contribution_sd", "High-exposure model: high flood"),
    ("high_joint_contribution_sd", "High-exposure model: joint high"),
]

selected_cols = [col for col, _ in selected_iir_panels + selected_mir_panels]
selected_shared_clim = selected_common_symmetric_limits(
    [selected_iir_map, selected_mir_map],
    selected_cols,
)

plot_selected_contribution_row(
    selected_iir_map,
    selected_iir_panels,
    "Standardized contributions: infection-to-incidence conversion",
    "figure2e",
    selected_shared_clim,
)

plot_selected_contribution_row(
    selected_mir_map,
    selected_mir_panels,
    "Standardized contributions: incidence-to-mortality conversion",
    "figure3e",
    selected_shared_clim,
)

selected_contribution_source = pd.concat(
    [
        selected_iir_map.drop(columns="geometry", errors="ignore"),
        selected_mir_map.drop(columns="geometry", errors="ignore"),
    ],
    ignore_index=True,
)

preview_cols = [
    "map_conversion",
    "adm2ID",
    "region",
    "cont_edi_contribution_sd",
    "high_edi_contribution_sd",
    "high_heat_contribution_sd",
    "cont_flood_contribution_sd",
    "high_flood_contribution_sd",
    "high_joint_contribution_sd",
]
preview_cols = [col for col in preview_cols if col in selected_contribution_source.columns]
display(selected_contribution_source[preview_cols].head())
